In [ ]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime

def okdiario_scraper(category):
    if category=="internacional":
        url = "https://okdiario.com/internacional/feed"
    elif category=="nacional":
        url = "https://okdiario.com/espana/feed"
    elif category=="cultura":
        url = "https://okdiario.com/cultura/feed"

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'xml')

    items = soup.find_all('item')
    print(f"Total items: {len(items)}")
    datos = []

    for item in items:
        # Extaer datos
        title = item.find("title").text if item.find("title") else None
        link = item.find("link").text if item.find("link") else None
        date = item.find("pubDate").text if item.find("pubDate") else None
        subtitle = item.find("description").text if item.find("description") else None
        #category = item.find("category").text if item.find("category") else None # Peude salir valencia, españa... mejor pongo nacional para unificarlo después
        
        # Formatear datos
        formatted_date =datetime.strptime(date, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d") # Limpiar fecha

        if subtitle:
            soup_subtitle = BeautifulSoup(subtitle, 'html.parser')
            for link_tag in soup_subtitle.find_all('a'):
                link_tag.decompose()
            subtitle = ' '.join(soup_subtitle.get_text(separator=' ', strip=True).split())

        # Elegir bien el contenido
        content_encoded_list = item.find_all("content:encoded")
        if content_encoded_list:
            html_content = content_encoded_list[-1].text
            soup_content = BeautifulSoup(html_content, 'html.parser')
            
            # Elimina los spans de comentarios
            for span in soup_content.find_all('span', class_='comment-text'):
                span.decompose()
            
            description = soup_content.get_text(separator=' ', strip=True)
        else:
            description = None
        # Guardar datos
        datos.append({
            "Link": link,
            "Periódico": "okdiario",
            "Fecha": formatted_date,
            "Título": title,
            "Subtítulo": subtitle,
            "Categoría": category,
            "Contenido": description
        })
    return(datos)


def guardar_sin_duplicados(nuevos_datos, archivo='data/okdiario.json'):
    # Cargar datos existentes
    try:
        with open(archivo, 'r', encoding='utf-8') as f:
            datos_existentes = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        print(f"Archivo corrupto o no existe. Iniciando desde cero.")
        datos_existentes = []
    
    # Links existentes (para evitar duplicados)
    links_existentes = {item['Link'] for item in datos_existentes}
    
    # Añadir solo nuevos
    nuevos_agregados = 0
    for item in nuevos_datos:
        if item['Link'] not in links_existentes:
            datos_existentes.append(item)
            links_existentes.add(item['Link'])
            nuevos_agregados += 1
    
    # Guardar
    with open(archivo, 'w', encoding='utf-8') as f:
        json.dump(datos_existentes, f, indent=2, ensure_ascii=False)
    
    print(f"Guardados {nuevos_agregados} items nuevos. Total en archivo: {len(datos_existentes)}")

In [26]:
# Extracción de los datos para las 3 categorías: internacional, nacional, cultura
datos_internacional=okdiario_scraper(category="internacional")
datos_nacional=okdiario_scraper(category="nacional")
datos_cultura=okdiario_scraper(category="cultura")

# Guardar datos
datos_totales = datos_cultura + datos_nacional + datos_internacional
guardar_sin_duplicados(datos_totales)

Total items: 50
Total items: 50
Total items: 50
Archivo corrupto o no existe. Iniciando desde cero.
Guardados 150 items nuevos. Total en archivo: 150
